# Audio Denoising Ablation: Whisper + TF-IDF Logistic Regression

This notebook measures the effect of audio preprocessing while keeping Whisper and the selected TF-IDF classifier unchanged.

- **Baseline:** raw audio → Whisper → TF-IDF + logistic regression
- **Improved:** trim silence + remove stationary noise + normalize → Whisper → the same TF-IDF + logistic regression model

The experiment compares Word Error Rate (WER), Character Error Rate (CER), predicted disease, and confidence. It also repeats the comparison after adding controlled synthetic noise.

## 1. Dependencies

Run the following installation command only if a package is missing.

In [1]:
# Uncomment if needed:
# %pip install librosa noisereduce soundfile jiwer pandas scikit-learn torch transformers

## 2. Imports and experiment settings

In [2]:
from pathlib import Path
import random
import re

import jiwer
import joblib
import librosa
import noisereduce as nr
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

SEED = 42
TARGET_SAMPLE_RATE = 16_000
SNR_LEVELS_DB = [20, 10, 5, 0]
MIN_CLASSIFICATION_CONFIDENCE = 0.50
REFERENCE_TRANSCRIPT = "I have a headache every day in the morning."

AUDIO_PATH = Path("WhatsApp Audio 2026-05-07 at 00.20.22.wav")
DATA_PATH = Path("Symptom2Disease.csv")
CLASSIFIER_MODEL_PATH = Path("tfidf_logistic_model.joblib")
WHISPER_MODEL_NAME = "openai/whisper-small"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 3. Load the recording and disease labels

`librosa` is used because the WhatsApp recording has a `.wav` filename but is not stored as a standard RIFF/WAV file.

In [3]:
for required_path in [AUDIO_PATH, DATA_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file or folder not found: {required_path.resolve()}")

raw_audio, sample_rate = librosa.load(
    AUDIO_PATH,
    sr=TARGET_SAMPLE_RATE,
    mono=True,
)
raw_audio = raw_audio.astype(np.float32)

dataset = pd.read_csv(DATA_PATH).dropna(subset=["label", "text"]).copy()
class_names = sorted(dataset["label"].unique().tolist())

print(f"Audio duration: {len(raw_audio) / sample_rate:.2f} seconds")
print(f"Sample rate: {sample_rate} Hz")
print(f"Disease classes: {len(class_names)}")
print("Reference transcript:", REFERENCE_TRANSCRIPT)

Audio duration: 3.86 seconds
Sample rate: 16000 Hz
Disease classes: 24
Reference transcript: I have a headache every day in the morning.


## 4. Load Whisper and train the selected text classifier

The TF-IDF vocabulary and logistic regression classifier are fitted once and then reused for every transcript. The full labeled dataset is used here to create the final prototype model after the held-out comparison was completed separately in `BERT.ipynb`.

In [4]:
whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_NAME)
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_MODEL_NAME
).to(device)
whisper_model.eval()

tfidf_logistic_model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                max_features=20_000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2_000,
                random_state=SEED,
            ),
        ),
    ]
)

tfidf_logistic_model.fit(
    dataset["text"].astype(str),
    dataset["label"].astype(str),
)
joblib.dump(tfidf_logistic_model, CLASSIFIER_MODEL_PATH)

print("Whisper model:", WHISPER_MODEL_NAME)
print("Selected classifier: TF-IDF + Logistic Regression")
print("Training descriptions:", len(dataset))
print("Saved classifier:", CLASSIFIER_MODEL_PATH.resolve())

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 4474.51it/s]


Whisper model: openai/whisper-small
Selected classifier: TF-IDF + Logistic Regression
Training descriptions: 1200
Saved classifier: C:\Year 3 Sem 3\Thesis\Denoise + Whisper\tfidf_logistic_model.joblib


## 5. Baseline and improved audio preparation

In [5]:
def prepare_baseline_audio(audio):
    """Return raw resampled audio without denoising or normalization."""
    return np.asarray(audio, dtype=np.float32)


def prepare_improved_audio(audio, sample_rate=TARGET_SAMPLE_RATE):
    """Trim silence, reduce stationary noise, remove DC offset, and normalize."""
    audio = np.asarray(audio, dtype=np.float32)
    audio = audio - np.mean(audio)

    trimmed, _ = librosa.effects.trim(audio, top_db=30)
    if trimmed.size == 0:
        trimmed = audio

    denoised = nr.reduce_noise(
        y=trimmed,
        sr=sample_rate,
        stationary=True,
        # Mild reduction preserves speech phonemes better than aggressive gating.
        prop_decrease=0.5,
    )

    peak = np.max(np.abs(denoised))
    if peak > 0:
        denoised = 0.95 * denoised / peak

    return denoised.astype(np.float32)


def add_noise_at_snr(audio, snr_db, rng):
    """Add zero-mean Gaussian noise at a requested signal-to-noise ratio."""
    audio = np.asarray(audio, dtype=np.float32)
    signal_power = np.mean(np.square(audio))
    if signal_power == 0:
        return audio.copy()

    noise = rng.normal(0.0, 1.0, size=audio.shape).astype(np.float32)
    noise_power = np.mean(np.square(noise))
    target_noise_power = signal_power / (10 ** (snr_db / 10))
    scaled_noise = noise * np.sqrt(target_noise_power / noise_power)
    noisy_audio = audio + scaled_noise

    peak = np.max(np.abs(noisy_audio))
    if peak > 1.0:
        noisy_audio = noisy_audio / peak

    return noisy_audio.astype(np.float32)

## 6. Shared transcription and classification functions

In [6]:
def transcribe_batch(audio_arrays, batch_size=4):
    """Transcribe audio arrays with fixed English Whisper settings."""
    transcripts = []

    for start in range(0, len(audio_arrays), batch_size):
        batch = audio_arrays[start : start + batch_size]
        inputs = whisper_processor(
            batch,
            sampling_rate=TARGET_SAMPLE_RATE,
            return_tensors="pt",
            padding=True,
            return_attention_mask=True,
        )
        input_features = inputs.input_features.to(device)
        attention_mask = inputs.attention_mask.to(device)

        with torch.inference_mode():
            predicted_ids = whisper_model.generate(
                input_features=input_features,
                attention_mask=attention_mask,
                language="en",
                task="transcribe",
            )

        transcripts.extend(
            whisper_processor.batch_decode(
                predicted_ids,
                skip_special_tokens=True,
            )
        )

    return [transcript.strip() for transcript in transcripts]


def normalize_transcript(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


def word_error_rate(reference, hypothesis):
    return jiwer.wer(
        normalize_transcript(reference),
        normalize_transcript(hypothesis),
    )


def character_error_rate(reference, hypothesis):
    return jiwer.cer(
        normalize_transcript(reference),
        normalize_transcript(hypothesis),
    )


def classify_transcript(text):
    probabilities = tfidf_logistic_model.predict_proba([text])[0]
    predicted_index = int(np.argmax(probabilities))
    predicted_disease = tfidf_logistic_model.classes_[predicted_index]
    confidence = float(probabilities[predicted_index])
    status = (
        "Accepted"
        if confidence >= MIN_CLASSIFICATION_CONFIDENCE
        else "Low confidence - seek professional medical advice"
    )
    return predicted_disease, confidence, status

## 7. Build the real-audio and controlled-noise test cases

The random generator is reset once, so rerunning the notebook creates the same noisy signals.

In [7]:
test_cases = []

def add_pipeline_pair(source_name, snr_db, source_audio):
    test_cases.append(
        {
            "Source": source_name,
            "SNR (dB)": snr_db,
            "Pipeline": "Baseline (raw)",
            "Audio": prepare_baseline_audio(source_audio),
        }
    )
    test_cases.append(
        {
            "Source": source_name,
            "SNR (dB)": snr_db,
            "Pipeline": "Improved (denoised)",
            "Audio": prepare_improved_audio(source_audio),
        }
    )


add_pipeline_pair("Real recording", "Original", raw_audio)

rng = np.random.default_rng(SEED)
for snr_db in SNR_LEVELS_DB:
    noisy_audio = add_noise_at_snr(raw_audio, snr_db, rng)
    add_pipeline_pair("Synthetic Gaussian noise", snr_db, noisy_audio)

print(f"Prepared {len(test_cases)} pipeline evaluations.")

Prepared 10 pipeline evaluations.


## 8. Run the ablation

In [8]:
audio_batch = [case["Audio"] for case in test_cases]
transcripts = transcribe_batch(audio_batch)

result_rows = []
for case, transcript in zip(test_cases, transcripts):
    predicted_disease, confidence, prediction_status = classify_transcript(transcript)
    result_rows.append(
        {
            "Source": case["Source"],
            "SNR (dB)": case["SNR (dB)"],
            "Pipeline": case["Pipeline"],
            "Transcript": transcript,
            "WER": word_error_rate(REFERENCE_TRANSCRIPT, transcript),
            "CER": character_error_rate(REFERENCE_TRANSCRIPT, transcript),
            "Predicted Disease": predicted_disease,
            "Confidence": confidence,
            "Prediction Status": prediction_status,
        }
    )

results = pd.DataFrame(result_rows)
results_percent = results.copy()
results_percent[["WER", "CER", "Confidence"]] = (
    results_percent[["WER", "CER", "Confidence"]].mul(100).round(2)
)

print("WER, CER, and confidence are displayed as percentages (%).")
display(results_percent)

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


WER, CER, and confidence are displayed as percentages (%).


,Source,SNR (dB),Pipeline,Transcript,WER,CER,Predicted Disease,Confidence,Prediction Status
0,Real recording,Original,Baseline (raw),I have a haydack every day in the morning.,11.11,9.52,Chicken pox,7.78,Low confidence - seek professional medical advice
1,Real recording,Original,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
2,Synthetic Gaussian noise,20,Baseline (raw),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
3,Synthetic Gaussian noise,20,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
4,Synthetic Gaussian noise,10,Baseline (raw),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
5,Synthetic Gaussian noise,10,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
6,Synthetic Gaussian noise,5,Baseline (raw),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
7,Synthetic Gaussian noise,5,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
8,Synthetic Gaussian noise,0,Baseline (raw),I have a headache every day.,33.33,35.71,Hypertension,16.62,Low confidence - seek professional medical advice
9,Synthetic Gaussian noise,0,Improved (denoised),I have a headache every day in a minute.,22.22,19.05,Hypertension,16.62,Low confidence - seek professional medical advice


## 9. Real recording comparison

In [9]:
real_audio_results = results_percent[
    results_percent["Source"] == "Real recording"
].reset_index(drop=True)

display(real_audio_results)

real_wer_change = real_audio_results.loc[0, "WER"] - real_audio_results.loc[1, "WER"]
real_cer_change = real_audio_results.loc[0, "CER"] - real_audio_results.loc[1, "CER"]

print(f"WER reduction from preprocessing: {real_wer_change:.2f} percentage points")
print(f"CER reduction from preprocessing: {real_cer_change:.2f} percentage points")
if real_wer_change < 0 or real_cer_change < 0:
    print("Note: preprocessing made at least one transcription metric worse on this recording.")

,Source,SNR (dB),Pipeline,Transcript,WER,CER,Predicted Disease,Confidence,Prediction Status
0,Real recording,Original,Baseline (raw),I have a haydack every day in the morning.,11.11,9.52,Chicken pox,7.78,Low confidence - seek professional medical advice
1,Real recording,Original,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice


WER reduction from preprocessing: 11.11 percentage points
CER reduction from preprocessing: 9.52 percentage points


## 10. Synthetic-noise robustness summary

In [10]:
noise_results = results[
    results["Source"] == "Synthetic Gaussian noise"
].copy()

robustness_summary = (
    noise_results.groupby("Pipeline", as_index=False)
    .agg(
        Mean_WER=("WER", "mean"),
        Mean_CER=("CER", "mean"),
        Mean_Confidence=("Confidence", "mean"),
        Unique_Disease_Predictions=("Predicted Disease", "nunique"),
        Low_Confidence_Predictions=(
            "Prediction Status",
            lambda values: sum(value != "Accepted" for value in values),
        ),
    )
)

robustness_summary_percent = robustness_summary.copy()
robustness_summary_percent[["Mean_WER", "Mean_CER", "Mean_Confidence"]] = (
    robustness_summary_percent[["Mean_WER", "Mean_CER", "Mean_Confidence"]]
    .mul(100)
    .round(2)
)

display(
    results_percent[results_percent["Source"] == "Synthetic Gaussian noise"]
    .sort_values(["SNR (dB)", "Pipeline"], ascending=[False, True])
    .reset_index(drop=True)
)
print("Average across the four synthetic noise levels:")
display(robustness_summary_percent)

,Source,SNR (dB),Pipeline,Transcript,WER,CER,Predicted Disease,Confidence,Prediction Status
0,Synthetic Gaussian noise,20,Baseline (raw),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
1,Synthetic Gaussian noise,20,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
2,Synthetic Gaussian noise,10,Baseline (raw),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
3,Synthetic Gaussian noise,10,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
4,Synthetic Gaussian noise,5,Baseline (raw),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
5,Synthetic Gaussian noise,5,Improved (denoised),I have a headache every day in the morning.,0.00,0.00,Hypertension,14.76,Low confidence - seek professional medical advice
6,Synthetic Gaussian noise,0,Baseline (raw),I have a headache every day.,33.33,35.71,Hypertension,16.62,Low confidence - seek professional medical advice
7,Synthetic Gaussian noise,0,Improved (denoised),I have a headache every day in a minute.,22.22,19.05,Hypertension,16.62,Low confidence - seek professional medical advice


Average across the four synthetic noise levels:


,Pipeline,Mean_WER,Mean_CER,Mean_Confidence,Unique_Disease_Predictions,Low_Confidence_Predictions
0,Baseline (raw),8.33,8.93,15.23,1,4
1,Improved (denoised),5.56,4.76,15.23,1,4


## 11. Save thesis-ready tables

In [11]:
results_percent.to_csv("audio_denoising_ablation_results.csv", index=False)

print("Saved:")
print("- audio_denoising_ablation_results.csv")
print("- tfidf_logistic_model.joblib")

Saved:
- audio_denoising_ablation_results.csv
- tfidf_logistic_model.joblib


## Thesis interpretation and limitation

Use this as a **pilot ablation study**. Lower WER and CER indicate better transcription. Higher classifier confidence is not automatically better unless the predicted disease is known to be correct.

Only one recording is available, and it has no confirmed disease label. Therefore, this experiment cannot validly calculate downstream disease-classification accuracy, precision, recall, or F1. A full thesis evaluation should include multiple recordings with verified transcripts and disease labels.